# Registration Fact Table — Gold Layer (`company_db.gold.registration_fact`)

This notebook builds and maintains the **Registration Fact** table in the Gold layer of the Company data warehouse. It captures every auction registration event per user and is the primary fact table for reporting on participation, approval rates, and auction activity.

---

## What This Notebook Does

1. Creates a staging view (`final_registrations`) that joins Silver registration records with the Gold Users Dimension to resolve surrogate keys
2. Initialises the `registration_fact` Delta table if it does not already exist
3. Merges staged records into the fact table (insert new registrations, update existing ones)

---

## Source Tables

| Table | Layer | Description |
|---|---|---|
| `company_db.silver.registeredusers` | Silver | Raw auction registration records (one row per user × auction) |
| `company_db.gold.users_dim` | Gold | User dimension; provides the `user_dim_key` surrogate key |

## Target Table

`company_db.gold.registration_fact`  
Stored in Azure Data Lake Storage Gen2: `abfss://auction@company.dfs.core.windows.net/gold/registration_fact`

---

## MERGE Key

Records are matched on the **composite natural key**: `user_id + auction_id`  
This ensures one fact row per user per auction.

---

## Column Reference

| Column | Type | Description |
|---|---|---|
| `user_dim_key` | BIGINT | **Foreign key** → `company_db.gold.users_dim.user_dim_key` |
| `user_id` | INTEGER | Source system user identifier |
| `auction_id` | INTEGER | Auction identifier |
| `registration_datetime` | TIMESTAMP | When the user registered for the auction |
| `date_approved` | TIMESTAMP | When the registration was approved (NULL if pending) |
| `registration_status` | STRING | Current registration status (e.g. `Approved`, `Pending`, `Rejected`) |
| `auto_approved` | STRING | Whether the system auto-approved the registration (`Y` / `N`) |
| `start_date` | DATE | Auction start date |
| `end_date` | DATE | Auction end date |
| `created_at` | TIMESTAMP | Timestamp when the row was first inserted |
| `updated_at` | TIMESTAMP | Timestamp of the last MERGE update |

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
spark.sql(f"""
CREATE VIEW IF NOT EXISTS  {catalog}.gold.final_registrations as 
(SELECT 
u.user_dim_key,
ru.user_id,
ru.auction_id,
ru.registration_datetime,
ru.date_approved,
ru.registration_status,
ru.auto_approved,
ru.start_date,
ru.end_date

From 
{catalog}.silver.registeredusers as ru
LEFT JOIN {catalog}.gold.users_dim as u
ON ru.user_id = u.user_id)""")

---
## Step 1 · Create Staging View (`final_registrations`)

Creates (or replaces) a **staging view** that joins:
- `company_db.silver.registeredusers` — the raw registration records
- `company_db.gold.users_dim` — the user dimension (LEFT JOIN on `user_id`)

The LEFT JOIN ensures no registration records are lost even if a user has not yet been loaded into the dimension table. The view resolves the `user_dim_key` surrogate key needed by the fact table and acts as the source for the MERGE in Step 3.

> **Note:** This is a logical view — it is recomputed on every MERGE run. No data is persisted at this step.

In [ ]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {catalog}.gold.registration_fact
(
  user_dim_key BIGINT,
  user_id INTEGER,
  auction_id INTEGER,
  registration_datetime TIMESTAMP,
  date_approved TIMESTAMP,
  registration_status STRING,
  auto_approved STRING,
  start_date DATE,
  end_date DATE,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold/registration_fact';""")

---
## Step 2 · Initialize Gold Fact Table

Creates the `company_db.gold.registration_fact` Delta table **if it does not already exist**. This is a one-time operation — on subsequent runs the statement is a no-op and no existing data is affected.

The table uses the **Delta format** for full ACID transaction support (MERGE, time travel, schema evolution). Backed by ADLS Gen2 at `gold/registration_fact`.

Granularity: **one row per `user_id` × `auction_id` pair** (enforced by the MERGE key in Step 3).

In [ ]:
spark.sql(
f"""
MERGE INTO {catalog}.gold.registration_fact AS target
USING {catalog}.gold.final_registrations AS source
ON target.user_id = source.user_id 
   AND target.auction_id = source.auction_id
WHEN MATCHED THEN
UPDATE SET
  target.registration_datetime = source.registration_datetime,
  target.date_approved = source.date_approved,
  target.registration_status = source.registration_status,
  target.auto_approved = source.auto_approved,
  target.start_date = source.start_date,
  target.end_date = source.end_date,
  target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
  user_dim_key,
  user_id,
  auction_id,
  registration_datetime,
  date_approved,
  registration_status,
  auto_approved,
  start_date,
  end_date,
  created_at,
  updated_at
)
VALUES (
  source.user_dim_key,
  source.user_id,
  source.auction_id,
  source.registration_datetime,
  source.date_approved,
  source.registration_status,
  source.auto_approved,
  source.start_date,
  source.end_date,
  current_timestamp(),
  current_timestamp()
)""")

---
## Step 3 · MERGE into Gold Fact Table

Applies the staged registrations from `final_registrations` into `registration_fact` using a two-condition match key: **`user_id + auction_id`**.

| Condition | Action | Notes |
|---|---|---|
| MATCHED | **UPDATE** registration fields + `updated_at = current_timestamp()` | Keeps status, approval, and date fields current |
| NOT MATCHED | **INSERT** all columns + `created_at = current_timestamp()`, `updated_at = current_timestamp()` | Adds new registration records |

The `user_dim_key` is only populated on INSERT (not updated on MATCH) to preserve the stable FK link to the dimension.

---
## Step 4 · Verification

Queries the final table to validate results. Spot-check that:
- Row count matches expected registrations
- `user_dim_key` is non-null (indicates successful dimension join)
- `created_at` and `updated_at` are populated for all rows

In [ ]:

# Verify final row count and check for missing dimension keys
spark.sql(f"""
SELECT
  COUNT(*)                                        AS total_registrations,
  COUNT(DISTINCT user_id)                         AS unique_users,
  COUNT(DISTINCT auction_id)                      AS unique_auctions,
  SUM(CASE WHEN user_dim_key IS NULL THEN 1 END)  AS missing_dim_keys,
  MIN(created_at)                                 AS earliest_created,
  MAX(updated_at)                                 AS latest_updated
FROM {catalog}.gold.registration_fact;""")

In [ ]:
%sql
-- DESCRIBE HISTORY company_db.gold.registration_fact;
-- RESTORE TABLE company_db.gold.registration_fact TO VERSION AS OF 1;